# EN3160 Assignment 03 — Neural Networks
**Abdul Rahman Mohamed Farharth | 230507R**

University of Moratuwa · Image Processing and Machine Vision

Three experiments: (1) manual CIFAR-10 dense network, (2) LeNet-5 versus a three-convolution CNN on MNIST, (3) ResNet18 fine-tuning versus feature extraction on the Hymenoptera dataset. Results and discussion should be finalized only after a clean run.

## Setup
Run from the `Assignment_03` directory in VS Code. Required packages: `pip install torch torchvision numpy matplotlib scikit-learn`. Data downloads require internet access. Training uses a CUDA GPU if present.

In [ ]:
from pathlib import Path
import random
import numpy as np
import torch
from torch import nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA = Path('data'); DATA.mkdir(exist_ok=True)
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
print('Device:', device, '| torch:', torch.__version__)

## Q1 — Dense network with manual forward and backward passes
The original one-layer CIFAR-10 network is extended with **100 sigmoid hidden nodes**. The output is a vector of ten logits, and cross-entropy is computed using softmax probabilities. All weight derivatives are calculated **manually**, without `loss.backward()` or PyTorch optimizers. Run for **10 epochs** and report training/test accuracy.

In [ ]:
cifar_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((.5,)*3, (.5,)*3)])
cifar_train = datasets.CIFAR10(DATA, train=True, download=True, transform=cifar_tf)
cifar_test = datasets.CIFAR10(DATA, train=False, download=True, transform=cifar_tf)
q1_train = DataLoader(cifar_train, batch_size=100, shuffle=True, num_workers=0)
q1_test = DataLoader(cifar_test, batch_size=256, shuffle=False, num_workers=0)
Din, H, K = 3072, 100, 10
w1 = torch.randn(Din, H, device=device) / np.sqrt(Din)
b1 = torch.zeros(H, device=device)
w2 = torch.randn(H, K, device=device) / np.sqrt(H)
b2 = torch.zeros(K, device=device)

def dense_forward(images):
    x = images.flatten(1)
    h = torch.sigmoid(x @ w1 + b1)
    z = h @ w2 + b2
    return x, h, z

@torch.no_grad()
def dense_accuracy(loader):
    correct = total = 0
    for images, targets in loader:
        _, _, z = dense_forward(images.to(device))
        correct += (z.argmax(1) == targets.to(device)).sum().item()
        total += len(targets)
    return 100 * correct / total

q1_history = {'loss':[], 'train_acc':[], 'test_acc':[]}
lr = .1
for epoch in range(10):
    total_loss = total_count = 0
    for images, targets in q1_train:
        targets = targets.to(device)
        x, h, logits = dense_forward(images.to(device))
        n = len(targets)
        probs = torch.softmax(logits, dim=1)
        loss = -torch.log(probs[torch.arange(n, device=device), targets].clamp_min(1e-12)).mean()
        dz = probs.clone()
        dz[torch.arange(n, device=device), targets] -= 1
        dz /= n
        dw2, db2 = h.T @ dz, dz.sum(0)
        dh = (dz @ w2.T) * h * (1-h)
        dw1, db1 = x.T @ dh, dh.sum(0)
        w1 -= lr * dw1; b1 -= lr * db1
        w2 -= lr * dw2; b2 -= lr * db2
        total_loss += loss.item() * n
        total_count += n
    q1_history['loss'].append(total_loss/total_count)
    q1_history['train_acc'].append(dense_accuracy(q1_train))
    q1_history['test_acc'].append(dense_accuracy(q1_test))
    print(f'Epoch {epoch+1:02d}/10 | loss {q1_history["loss"][-1]:.4f} | train {q1_history["train_acc"][-1]:.2f}% | test {q1_history["test_acc"][-1]:.2f}%')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(range(1,11), q1_history['loss']); ax[0].set(title='Cross-entropy', xlabel='Epoch', ylabel='Loss')
ax[1].plot(range(1,11), q1_history['train_acc'], label='Train')
ax[1].plot(range(1,11), q1_history['test_acc'], label='Test')
ax[1].legend(); ax[1].set(title='Classification accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
plt.tight_layout(); plt.savefig(OUT/'q1_curves.png', dpi=150); plt.show()

**Q1 discussion (fill after execution):** Report the epoch-10 training and test accuracies. Discuss the gap and why flattening the pixels ignores spatial locality. No numbers are assumed in advance.

## Q2 — LeNet-5 vs modern CNN on MNIST
Use the same train/test split, input normalization, Adam optimizer, learning rate, batch sizes and **10 epochs** for both architectures. LeNet-5 here is an adapted 28×28 implementation with tanh and average pooling. The modern CNN uses **three convolutional layers**, batch normalization and dropout.

In [ ]:
mnist_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((.1307,), (.3081,))])
mnist_train = datasets.MNIST(DATA, train=True, download=True, transform=mnist_tf)
mnist_test = datasets.MNIST(DATA, train=False, download=True, transform=mnist_tf)
train_mnist = DataLoader(mnist_train, batch_size=128, shuffle=True, num_workers=0)
test_mnist = DataLoader(mnist_test, batch_size=256, shuffle=False, num_workers=0)

class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1,6,5,padding=2), nn.Tanh(), nn.AvgPool2d(2),
                                      nn.Conv2d(6,16,5), nn.Tanh(), nn.AvgPool2d(2))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(16*5*5,120), nn.Tanh(),
                                        nn.Linear(120,84), nn.Tanh(), nn.Linear(84,10))
    def forward(self,x): return self.classifier(self.features(x))

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
                                    nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),
                                    nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),
                                    nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(.3),nn.Linear(128,10))
    def forward(self,x): return self.layers(x)

def evaluate(model,loader,criterion,collect=False):
    model.eval(); total_loss = correct = total = 0
    true, pred = [], []
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            scores = model(x); p = scores.argmax(1)
            total_loss += criterion(scores,y).item()*len(y)
            correct += (p==y).sum().item(); total += len(y)
            if collect: true.extend(y.cpu().tolist()); pred.extend(p.cpu().tolist())
    return (total_loss/total,100*correct/total,true,pred)

def train_supervised(model,train_loader,test_loader,epochs=10,lr=.001):
    model=model.to(device); loss_fn=nn.CrossEntropyLoss()
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    hist={'train_loss':[],'train_acc':[],'test_loss':[],'test_acc':[]}
    for epoch in range(epochs):
        model.train(); count=correct=0; loss_total=0
        for x,y in train_loader:
            x,y=x.to(device),y.to(device)
            opt.zero_grad(set_to_none=True)
            scores=model(x); loss=loss_fn(scores,y)
            loss.backward(); opt.step()
            count+=len(y); correct+=(scores.argmax(1)==y).sum().item(); loss_total+=loss.item()*len(y)
        test_loss,test_acc,_,_=evaluate(model,test_loader,loss_fn)
        for key,val in [('train_loss',loss_total/count),('train_acc',100*correct/count),
                        ('test_loss',test_loss),('test_acc',test_acc)]: hist[key].append(val)
        print(f'{epoch+1:02d}/{epochs} train {hist["train_acc"][-1]:.2f}% | test {test_acc:.2f}%')
    return model,hist

def plot_history(h,title,file):
    t=range(1,len(h['train_acc'])+1)
    fig,ax=plt.subplots(1,2,figsize=(11,4))
    for metric,axis in [('loss',ax[0]),('acc',ax[1])]:
        axis.plot(t,h['train_'+metric],label='Train')
        axis.plot(t,h['test_'+metric],label='Test')
        axis.set(xlabel='Epoch',ylabel='Loss' if metric=='loss' else 'Accuracy (%)'); axis.legend()
    fig.suptitle(title); plt.tight_layout(); plt.savefig(OUT/file,dpi=150); plt.show()

In [ ]:
q2_results={}
for name,model_type in [('LeNet5',LeNet5),('ModernCNN',SmallCNN)]:
    print('\nTraining',name)
    torch.manual_seed(SEED)
    net,hist=train_supervised(model_type(),train_mnist,test_mnist,epochs=10)
    q2_results[name]=(net,hist)
    plot_history(hist,name,'q2_'+name.lower()+'_curves.png')
    _,acc,truth,pred=evaluate(net,test_mnist,nn.CrossEntropyLoss(),collect=True)
    fig,ax=plt.subplots(figsize=(7,6))
    ConfusionMatrixDisplay(confusion_matrix(truth,pred,labels=list(range(10))),display_labels=list(range(10))).plot(ax=ax,cmap='Blues',colorbar=False)
    ax.set_title(name+' MNIST test confusion matrix')
    plt.tight_layout();plt.savefig(OUT/('q2_'+name.lower()+'_confusion.png'),dpi=150);plt.show()
    print('Final test accuracy:',f'{acc:.2f}%')

**Q2 discussion (fill after execution):** Compare final accuracies and losses, examine training–test gaps, and identify the digits most often confused from the matrices. Training accuracy is aggregated during parameter updates; the test metric is calculated using the end-of-epoch model.

## Q3 — ResNet18 transfer learning on Hymenoptera
Use pretrained ImageNet1K ResNet18 with two output classes (ants, bees). Compare **fine-tuning** all layers against **feature extraction** (freeze backbone and train final FC layer). The official tutorial archive has `train` and `val` splits; call the latter *validation* rather than claiming an independent test set. The assignment does not specify an epoch count for Q3: we choose 10 for consistent comparison.

In [ ]:
import urllib.request, zipfile
hym_root=DATA/'hymenoptera_data'
if not hym_root.exists():
    archive=DATA/'hymenoptera_data.zip'
    urllib.request.urlretrieve('https://download.pytorch.org/tutorial/hymenoptera_data.zip',archive)
    with zipfile.ZipFile(archive) as z:
        root=DATA.resolve()
        for item in z.infolist():
            target=(DATA/item.filename).resolve()
            if root!=target and root not in target.parents: raise ValueError('Unsafe archive path')
        z.extractall(DATA)
assert (hym_root/'train').exists() and (hym_root/'val').exists()
normalize=transforms.Normalize([.485,.456,.406],[.229,.224,.225])
train_tf=transforms.Compose([transforms.RandomResizedCrop(224),transforms.RandomHorizontalFlip(),transforms.ToTensor(),normalize])
val_tf=transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),transforms.ToTensor(),normalize])
hym_train=datasets.ImageFolder(hym_root/'train',transform=train_tf)
hym_val=datasets.ImageFolder(hym_root/'val',transform=val_tf)
assert hym_train.classes==hym_val.classes
train_hym=DataLoader(hym_train,batch_size=16,shuffle=True,num_workers=0)
val_hym=DataLoader(hym_val,batch_size=32,shuffle=False,num_workers=0)
print(hym_train.classes,len(hym_train),'train',len(hym_val),'validation')

def make_resnet(frozen):
    model=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    if frozen:
        for param in model.parameters(): param.requires_grad=False
    model.fc=nn.Linear(model.fc.in_features,2)
    return model

def train_resnet(model,frozen,epochs=10):
    model=model.to(device)
    loss_fn=nn.CrossEntropyLoss()
    optimizer=torch.optim.SGD((p for p in model.parameters() if p.requires_grad),lr=.001,momentum=.9)
    history={'train_loss':[],'train_acc':[],'test_loss':[],'test_acc':[]}
    for epoch in range(epochs):
        model.train()
        if frozen:
            # Prevent updates to frozen batch-normalization running statistics.
            model.eval();model.fc.train()
        total=correct=0;loss_sum=0
        for x,y in train_hym:
            x,y=x.to(device),y.to(device)
            optimizer.zero_grad(set_to_none=True)
            z=model(x);loss=loss_fn(z,y)
            loss.backward();optimizer.step()
            total+=len(y);correct+=(z.argmax(1)==y).sum().item();loss_sum+=loss.item()*len(y)
        val_loss,val_acc,_,_=evaluate(model,val_hym,loss_fn)
        for key,value in [('train_loss',loss_sum/total),('train_acc',100*correct/total),
                          ('test_loss',val_loss),('test_acc',val_acc)]: history[key].append(value)
        print(f'{epoch+1:02d}/{epochs}: train {100*correct/total:.2f}% val {val_acc:.2f}%')
    return model,history

In [ ]:
q3_results={}
for title,frozen in [('Finetuning',False),('FeatureExtraction',True)]:
    print('\n'+title)
    torch.manual_seed(SEED)
    model,hist=train_resnet(make_resnet(frozen),frozen,epochs=10)
    q3_results[title]=(model,hist)
    plot_history(hist,title,'q3_'+title.lower()+'_curves.png')
    print(f'{title}: train {hist["train_acc"][-1]:.2f}%, validation {hist["test_acc"][-1]:.2f}%')

## Final results and interpretation (complete after running)

Record **actual** epoch-10 accuracy and loss for each experiment. For Q3 compare fine-tuning vs feature extraction while noting the small dataset and potential overfitting. Include generated figures and results in the PDF export of the notebook. The handout repeats an `a02` filename suffix for Assignment 3; confirm whether the lecturer wants `230507R_a03.pdf`.

GitHub: https://github.com/Abdul-Rahman-bme/EN3160-Image-processing-and-Machine-Vision